# 01_Data_Exploration.ipynb

## **0. Giới thiệu**

Notebook này thực hiện khám phá dữ liệu thô trên bộ dữ liệu [HR Analytics: Job Change of Data Scientists](https://www.kaggle.com/datasets/arashnic/hr-analytics-job-change-of-data-scientists?select=aug_train.csv). Mục tiêu:

- Đọc và tải dữ liệu dạng thô.

- Kiểm tra missing values.

- Phân loại các cột theo kiểu dữ liệu.

- Thống kê mô tả cho numeric/categorical.

- Phát hiện outlier (chỉ nhận diện, không chỉnh sửa).

- Rút ra nhận xét tổng quan trước khi bước sang Preprocessing.

## **1. Import thư viện & Load dữ liệu thô**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

project_root = os.path.abspath("..")
sys.path.append(project_root)

from src.data_processing import load_csv_numpy, missing_mask
from src.visualization import plot_hist_numeric

sns.set(style='whitegrid', rc={'figure.figsize': (8,5)})

In [3]:
headers, data = load_csv_numpy('../data/raw/aug_train.csv')
print(f'Tên các cột:')
for h in headers:
    print(f'- {h}')
print(f'\n(Số dòng, Số cột): {data.shape}')

Tên các cột:
- enrollee_id
- city
- city_development_index
- gender
- relevent_experience
- enrolled_university
- education_level
- major_discipline
- experience
- company_size
- company_type
- last_new_job
- training_hours
- target

(Số dòng, Số cột): (19158, 14)


In [ ]:
print(data[:5], '\n')

[['8949' 'city_103' '0.92' 'Male' 'Has relevent experience'
  'no_enrollment' 'Graduate' 'STEM' '>20' '' '' '1' '36' '1.0']
 ['29725' 'city_40' '0.7759999999999999' 'Male' 'No relevent experience'
  'no_enrollment' 'Graduate' 'STEM' '15' '50-99' 'Pvt Ltd' '>4' '47'
  '0.0']
 ['11561' 'city_21' '0.624' '' 'No relevent experience'
  'Full time course' 'Graduate' 'STEM' '5' '' '' 'never' '83' '0.0']
 ['33241' 'city_115' '0.789' '' 'No relevent experience' '' 'Graduate'
  'Business Degree' '<1' '' 'Pvt Ltd' 'never' '52' '1.0']
 ['666' 'city_162' '0.767' 'Male' 'Has relevent experience'
  'no_enrollment' 'Masters' 'STEM' '>20' '50-99' 'Funded Startup' '4' '8'
  '0.0']] 



## **2. Kiểm tra missing values theo từng cột**

In [ ]:
# đếm missing cho từng cột
missing_counts = np.array([
    missing_mask(data[:, i]).sum()
    for i in range(data.shape[1])
])

# tổng số dòng
total_rows = data.shape[0]

# tính tỉ lệ %
missing_percent = (missing_counts / total_rows) * 100

print('Đếm số dữ liệu bị thiếu của từng cột:')
for h, m, p in zip(headers, missing_counts, missing_percent):
    print(f'- {h:23} : {m:4d}  |  {p:6.2f}%')


Đếm số dữ liệu bị thiếu của từng cột:
- enrollee_id             :    0  |    0.00%
- city                    :    0  |    0.00%
- city_development_index  :    0  |    0.00%
- gender                  : 4508  |   23.53%
- relevent_experience     :    0  |    0.00%
- enrolled_university     :  386  |    2.01%
- education_level         :  460  |    2.40%
- major_discipline        : 2813  |   14.68%
- experience              :   65  |    0.34%
- company_size            : 5938  |   30.99%
- company_type            : 6140  |   32.05%
- last_new_job            :  423  |    2.21%
- training_hours          :    0  |    0.00%
- target                  :    0  |    0.00%


## **3. Phân loại cột theo kiểu dữ liệu (numeric / categorical)**

In [ ]:
# hàm kiểm tra cột có phải numeric hay không
def col_is_numeric(col):
    col_str = col.astype(str)
    
    # bỏ các giá trị bị missing ra
    non_missing = col_str[~missing_mask(col_str)]

    # thử convert toàn bộ sang float 1 lần
    try:
        non_missing.astype(float)
        return True
    except:
        check_num = np.frompyfunc(lambda x: x.replace('.', '', 1).isdigit(), 1, 1)
        is_num = check_num(non_missing).astype(bool)
        return is_num.mean() > 0.8

# tách danh sách numeric vs categorical
numeric_cols = []
categorical_cols = []

for i, h in enumerate(headers):
    if col_is_numeric(data[:, i]):
        numeric_cols.append(h)
    else:
        categorical_cols.append(h)

print('Những cột có kiểu dữ liệu numeric:')
for col in numeric_cols:
    print('-', col)

print('\nNhững cột có kiểu dữ liệu categorical:')
for col in categorical_cols:
    print('-', col)



Những cột có kiểu dữ liệu numeric:
- enrollee_id
- city_development_index
- experience
- training_hours
- target

Những cột có kiểu dữ liệu categorical:
- city
- gender
- relevent_experience
- enrolled_university
- education_level
- major_discipline
- company_size
- company_type
- last_new_job


## **4. Thống kê mô tả cho biến numeric**

In [6]:
# xem xét dataset gốc, thấy ở cột 'experience' có các dữ liệu '>20' 
# nên em sẽ xử lý TẠM THỜI phần dữ liệu này ở đây.

# chuẩn hóa cột experience để phục vụ thống kê EDA
exp_raw = data[:, headers.tolist().index('experience')].astype(str)

mask_exp_miss = missing_mask(exp_raw)

# copy để xử lý
exp_clean = exp_raw.copy()

# TẠM THỜI thay '<1' → '0'
exp_clean = np.where(exp_clean == '<1', '0', exp_clean)

# TẠM THỜI thay '>20' → '21'
exp_clean = np.where(exp_clean == '>20', '21', exp_clean)

# loại missing rồi convert
exp_numeric = exp_clean[~mask_exp_miss].astype(float)


In [9]:
# thống kê mô tả cho cột numeric

for h in numeric_cols:
    col = data[:, headers.tolist().index(h)]

    # xử lý cột đặc biệt
    if h == 'experience':
        arr = exp_numeric
    elif h == 'enrollee_id' or h == 'target':
        continue
    else:
        arr = col[~missing_mask(col)].astype(float)

    print(f'--- {h} ---')
    print(f'  Số lượng giá trị hợp lệ: {arr.size}')
    print(f'  Min     : {np.min(arr):.4f}')
    print(f'  Max     : {np.max(arr):.4f}')
    print(f'  Mean    : {np.mean(arr):.4f}')
    print(f'  Median  : {np.median(arr):.4f}')
    print(f'  Std     : {np.std(arr):.4f}')
    print()

--- city_development_index ---
  Số lượng giá trị hợp lệ: 19158
  Min     : 0.4480
  Max     : 0.9490
  Mean    : 0.8288
  Median  : 0.9030
  Std     : 0.1234

--- experience ---
  Số lượng giá trị hợp lệ: 19093
  Min     : 0.0000
  Max     : 21.0000
  Mean    : 10.1001
  Median  : 9.0000
  Std     : 6.7768

--- training_hours ---
  Số lượng giá trị hợp lệ: 19158
  Min     : 1.0000
  Max     : 336.0000
  Mean    : 65.3669
  Median  : 47.0000
  Std     : 60.0569



## **5. Thống kê cho biến categorical (Top 5 giá trị phổ biến)**

In [ ]:
TOP_K = 5

for h in categorical_cols:
    print(f'--- {h} ---')

    col = data[:, headers.tolist().index(h)].astype(str)

    # bỏ giá trị missing
    mask = ~missing_mask(col)
    col_nonmiss = col[mask]

    total_nonmiss = col_nonmiss.size  # tổng giá trị hợp lệ

    # đếm frequency — vectorized
    vals, counts = np.unique(col_nonmiss, return_counts=True)

    # sắp xếp giảm dần
    order = np.argsort(-counts)
    vals = vals[order]
    counts = counts[order]

    # lấy top K
    k = min(TOP_K, len(vals))
    top_vals = vals[:k]
    top_counts = counts[:k]

    # tính tỉ lệ %
    top_percent = (top_counts / total_nonmiss) * 100

    # in ra danh sách
    for i in range(k):
        print(f'  {i+1}. {top_vals[i]:20} — {top_counts[i]:5d}  ({top_percent[i]:5.2f}%)')

    print()


--- city ---
  1. city_103             —  4355  (22.73%)
  2. city_21              —  2702  (14.10%)
  3. city_16              —  1533  ( 8.00%)
  4. city_114             —  1336  ( 6.97%)
  5. city_160             —   845  ( 4.41%)

--- gender ---
  1. Male                 — 13221  (90.25%)
  2. Female               —  1238  ( 8.45%)
  3. Other                —   191  ( 1.30%)

--- relevent_experience ---
  1. Has relevent experience — 13792  (71.99%)
  2. No relevent experience —  5366  (28.01%)

--- enrolled_university ---
  1. no_enrollment        — 13817  (73.60%)
  2. Full time course     —  3757  (20.01%)
  3. Part time course     —  1198  ( 6.38%)

--- education_level ---
  1. Graduate             — 11598  (62.03%)
  2. Masters              —  4361  (23.32%)
  3. High School          —  2017  (10.79%)
  4. Phd                  —   414  ( 2.21%)
  5. Primary School       —   308  ( 1.65%)

--- major_discipline ---
  1. STEM                 — 14492  (88.66%)
  2. Humanities      

## **6. Phát hiện Outlier (chỉ nhận diện)**

In [12]:
# phát hiện outlier cho các cột numeric

for h in numeric_cols:
    col = data[:, headers.tolist().index(h)]

    # xử lý special cases: experience dùng bản numeric clean
    if h == 'experience':
        arr = exp_numeric
    elif h == 'enrollee_id' or h == 'target':
        continue
    else:
        arr = col[~missing_mask(col)].astype(float)

    # tính Q1, Q3
    Q1 = np.percentile(arr, 25)
    Q3 = np.percentile(arr, 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # mask outlier
    outlier_mask = (arr < lower_bound) | (arr > upper_bound)
    num_outliers = np.sum(outlier_mask)

    print(f'--- {h} ---')
    print(f'  Số lượng outlier: {num_outliers}')
    print(f'  Tỉ lệ outlier: {num_outliers / arr.size * 100:.2f}%')
    print(f'  Ngưỡng dưới: {lower_bound:.4f}')
    print(f'  Ngưỡng trên: {upper_bound:.4f}\n')


--- city_development_index ---
  Số lượng outlier: 17
  Tỉ lệ outlier: 0.09%
  Ngưỡng dưới: 0.4700
  Ngưỡng trên: 1.1900

--- experience ---
  Số lượng outlier: 0
  Tỉ lệ outlier: 0.00%
  Ngưỡng dưới: -14.0000
  Ngưỡng trên: 34.0000

--- training_hours ---
  Số lượng outlier: 984
  Tỉ lệ outlier: 5.14%
  Ngưỡng dưới: -74.5000
  Ngưỡng trên: 185.5000



## **7. Nhận xét tổng quan dữ liệu**

- Một số cột **categorical** có tỷ lệ thiếu khá cao như `gender` (~23%), `major_discipline` (~15%), `company_size` (~31%) và `company_type` (~32%), cho thấy thông tin hồ sơ ứng viên không đầy đủ và cần xử lý missing ở bước preprocessing.

- Các biến **numeric** nhìn chung sạch, nhưng phân phối đều bị skewed, đặc biệt là `training_hours` có đuôi phải (số thập phân) dài và xuất hiện nhiều giá trị lớn bất thường (~5% outlier). `city_development_index` tập trung ở mức cao, phản ánh phần lớn ứng viên đến từ các thành phố phát triển.

- Một số **categorical** phân phối không đồng đều, ví dụ `gender` nghiêng mạnh về **Male**; `city`, `education_level`, `enrolled_university` đều bị chi phối bởi một số nhóm giá trị chính, thể hiện sự mất cân bằng tự nhiên trong dữ liệu.

- Ngoại trừ `training_hours`, các biến **numeric** khác hầu như không có outlier đáng kể. Tuy nhiên, phân phối lệch và sự không cân bằng giữa các nhóm giá trị cho thấy dữ liệu cần được chuẩn hoá và xử lý lại trước khi đưa vào mô hình.

In [30]:
np.save('../data/processed/data_raw.npy', data)
np.save('../data/processed/headers.npy', headers)